# Advanced Pandas Transformations
**VEDA Technology Internship - Data Science Track - Task 1 (Level 3)**

**Objective:** Develop advanced data manipulation skills for complex data science workflows.

**Dataset:** Online Retail Dataset (UCI) - ~541k transactions from a UK-based online gift retailer.

**What this notebook covers:** eleven advanced transformations using `map()`, `replace()`, `apply()` (Series and row-wise),
`transform()`, vectorised conditional logic, binning, custom functions with `pipe()`, method chaining with `assign()`,
elementwise `DataFrame.map()`, and a performance benchmark of `apply()` vs vectorisation.

Each section states **what** the transformation does, **why** that method was chosen, and **what the result means**.

---
## 0. Setup and data loading

Download `Online Retail.xlsx` from the UCI Machine Learning Repository (or the Kaggle mirror as `data.csv`)
and place it in a `data/` folder next to this notebook. The loader below tries several filenames and falls back
to a reproducible synthetic sample with the same schema so the notebook always runs end to end.

In [1]:
import time
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

print("pandas", pd.__version__, "| numpy", np.__version__)

pandas 3.0.5 | numpy 2.5.2


In [2]:
def load_online_retail():
    """Load the UCI Online Retail dataset, falling back to a synthetic sample."""
    candidates = [
        ("data/Online Retail.xlsx", "excel"),
        ("data/OnlineRetail.xlsx", "excel"),
        ("data/data.csv", "csv"),
        ("data/OnlineRetail.csv", "csv"),
    ]
    for path, kind in candidates:
        if Path(path).exists():
            df = pd.read_excel(path) if kind == "excel" else pd.read_csv(path, encoding="ISO-8859-1")
            print(f"Loaded real dataset from {path}")
            return df

    print("Real dataset not found - generating a synthetic sample with the same schema.")
    rng = np.random.default_rng(7)
    n = 20_000
    countries = ["United Kingdom", "germany", "France", "EIRE", "Spain",
                 "Netherlands", "Australia", "USA", "Unspecified"]
    stock = [str(c) for c in rng.integers(10000, 99999, 400)]
    desc = ["WHITE HANGING HEART T-LIGHT HOLDER", " regency cakestand 3 tier ",
            "JUMBO BAG RED RETROSPOT", "PARTY BUNTING", "LUNCH BAG RED RETROSPOT",
            "?", "damaged", "ASSORTED COLOUR BIRD ORNAMENT"]
    df = pd.DataFrame({
        "InvoiceNo": rng.integers(536365, 581587, n).astype(str),
        "StockCode": rng.choice(stock, n),
        "Description": rng.choice(desc, n),
        "Quantity": rng.integers(-5, 60, n),
        "InvoiceDate": pd.to_datetime("2010-12-01") + pd.to_timedelta(rng.integers(0, 365 * 24, n), unit="h"),
        "UnitPrice": np.round(rng.gamma(2, 2.5, n), 2),
        "CustomerID": rng.choice([np.nan] + list(range(12346, 12800)), n),
        "Country": rng.choice(countries, n),
    })
    df.loc[rng.choice(n, 800, replace=False), "Description"] = np.nan
    return df


raw = load_online_retail()
print(raw.shape)
raw.head()

Loaded real dataset from data/Online Retail.xlsx
(541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [3]:
raw.info()
raw.isna().sum()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 33.1+ MB


InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

---
## 1. Baseline cleaning

Transformations are only meaningful on trustworthy rows, so first: normalise text, drop rows without a
`CustomerID` (they cannot be attributed to a customer), and remove cancellations / returns
(`Quantity <= 0`) and zero-priced rows. `Revenue` is then derived as the key numeric column.

In [4]:
df = raw.copy()

df["Description"] = df["Description"].str.strip().str.title()
df["Country"] = df["Country"].str.strip().str.title()

before = len(df)
df = df.dropna(subset=["CustomerID"])
df["CustomerID"] = df["CustomerID"].astype(int)
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)].reset_index(drop=True)

df["Revenue"] = df["Quantity"] * df["UnitPrice"]

print(f"Rows before: {before:,} -> after: {len(df):,}  ({before - len(df):,} removed)")
df.head()

Rows before: 541,909 -> after: 397,884  (144,025 removed)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
0,536365,85123A,White Hanging Heart T-Light Holder,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30
1,536365,71053,White Metal Lantern,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
2,536365,84406B,Cream Cupid Hearts Coat Hanger,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00
3,536365,84029G,Knitted Union Flag Hot Water Bottle,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
4,536365,84029E,Red Woolly Hottie White Heart.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34


---
## Transformation 1 - `Series.map()` for a dictionary lookup

**What:** collapse ~38 country names into four analytical regions.

**Why `map()`:** it is a Series-level, one-to-one lookup. `map()` is implemented as a fast hash lookup and is the
idiomatic choice here; `apply()` would call a Python function once per row for no benefit.

**Note:** `map()` returns `NaN` for keys that are missing from the dictionary, so `fillna("Other")` makes the
fallback explicit rather than silent.

In [25]:
region_map = {
    "United Kingdom": "UK", "Eire": "EU", "Germany": "EU", "France": "EU", "Spain": "EU",
    "Netherlands": "EU", "Belgium": "EU", "Switzerland": "EU", "Portugal": "EU", "Italy": "EU",
    "Finland": "EU", "Norway": "EU", "Sweden": "EU", "Denmark": "EU", "Austria": "EU",
    "Poland": "EU", "Greece": "EU", "Cyprus": "EU", "Czech Republic": "EU", "Lithuania": "EU",
    "Malta": "EU", "Iceland": "EU", "Channel Islands": "EU",
    "Australia": "APAC", "Japan": "APAC", "Singapore": "APAC",
    "Usa": "Americas", "Canada": "Americas", "Brazil": "Americas",    "Lebanon": "Other",
    "United Arab Emirates": "Other","Saudi Arabia": "Other", "European Community": "Other",
}

df["Region"] = df["Country"].map(region_map).fillna("Other")

print(df["Region"].value_counts())
df[["Country", "Region"]].drop_duplicates().head(10)

Region
UK          354321
EU           40728
APAC          1725
Other          748
Americas       362
Name: count, dtype: int64


,Country,Region
0,United Kingdom,UK
26,France,EU
195,Australia,APAC
376,Netherlands,EU
1098,Germany,EU
1225,Norway,EU
1393,Eire,EU
4035,Switzerland,EU
4250,Spain,EU
4437,Poland,EU


In [24]:
df["Country"].unique()

<StringArray>
[      'United Kingdom',               'France',            'Australia',          'Netherlands',              'Germany',               'Norway',
                 'Eire',          'Switzerland',                'Spain',               'Poland',             'Portugal',                'Italy',
              'Belgium',            'Lithuania',                'Japan',              'Iceland',      'Channel Islands',              'Denmark',
               'Cyprus',               'Sweden',              'Finland',              'Austria',               'Greece',            'Singapore',
              'Lebanon', 'United Arab Emirates',               'Israel',         'Saudi Arabia',       'Czech Republic',               'Canada',
              'Unknown',               'Brazil',                  'Usa',   'European Community',              'Bahrain',                'Malta',
                  'Rsa']
Length: 37, dtype: str

---
## Transformation 2 - `replace()` for value standardisation

**What:** turn placeholder junk (`"?"`, `"Damaged"`, `"Unspecified"`) into proper missing values or a
consistent label, including a regex-based rule.

**Why `replace()`:** it handles many-to-one and pattern-based substitution across a Series or the whole
DataFrame in a single call, which `map()` cannot do without listing every key.

In [6]:
junk = {"?": np.nan, "Damaged": np.nan, "Check": np.nan, "Missing": np.nan, "": np.nan}
df["Description"] = df["Description"].replace(junk)

df["Country"] = df["Country"].replace(r"(?i)^unspecified$", "Unknown", regex=True)

print("Descriptions now missing:", df["Description"].isna().sum())
print(df["Country"].value_counts().head(8))

Descriptions now missing: 0
Country
United Kingdom    354321
Germany             9040
France              8341
Eire                7236
Spain               2484
Netherlands         2359
Belgium             2031
Switzerland         1841
Name: count, dtype: int64


---
## Transformation 3 - `apply()` on a Series with a custom function

**What:** assign each product a price band from its unit price.

**Why `apply()`:** the rule is a branching Python function with no direct vectorised equivalent that stays
readable. This is a legitimate use of `apply()` - the logic is genuinely element-by-element.

In [7]:
df["UnitPrice"].describe()

count    397884.000000
mean          3.116488
std          22.097877
min           0.001000
25%           1.250000
50%           1.950000
75%           3.750000
max        8142.750000
Name: UnitPrice, dtype: float64

In [23]:
def price_band(price: float) -> str:
    """Bucket a unit price into a commercial band."""
    if price < 1.25:
        return "Budget"
    if price < 3.75:
        return "Standard"
    if price < 15:
        return "Premium"
    return "Luxury"


df["PriceBand"] = df["UnitPrice"].apply(price_band)

print(df["PriceBand"].value_counts())
df.groupby("PriceBand", observed=True)["Revenue"].agg(["count", "mean", "sum"]).round(2)

PriceBand
Standard    198884
Premium     100391
Budget       94878
Luxury        3731
Name: count, dtype: int64


,count,mean,sum
PriceBand,,,
Budget,94878,14.02,1330587.47
Luxury,3731,80.44,300109.35
Premium,100391,28.72,2883183.46
Standard,198884,22.11,4397527.62


---
## Transformation 4 - Row-wise `apply(axis=1)` across multiple columns

**What:** classify each order line using two columns at once (quantity and price together).

**Why row-wise `apply()`:** the decision needs several fields in the same expression. It is the slowest pattern
in pandas because every row becomes a Series object, so it is used only where the logic truly is cross-column -
and Transformation 6 shows the vectorised alternative for cases where it is not.

In [9]:
def classify_order(row: pd.Series) -> str:
    """Label an order line from quantity, price and revenue together."""
    if row["Quantity"] >= 24 and row["UnitPrice"] < 2:
        return "Bulk-Cheap"
    if row["Revenue"] > 200:
        return "High-Value"
    if row["Quantity"] <= 2 and row["UnitPrice"] >= 15:
        return "Single-Premium"
    return "Normal"


df["OrderType"] = df.apply(classify_order, axis=1)

print(df["OrderType"].value_counts())

OrderType
Normal            346653
Bulk-Cheap         45789
Single-Premium      2921
High-Value          2521
Name: count, dtype: int64


---
## Transformation 5 - `groupby().transform()` for aligned group statistics

**What:** attach each customer's lifetime spend to every one of their rows, then express each line as a share
of that total; and z-score every line against its own country.

**Why `transform()`:** it returns a result with **the same index and length as the input**, so it can be assigned
straight back as a column. `agg()` would collapse the frame to one row per group and force a merge instead.
This is the single most important idea in the task.

In [10]:
df["CustomerTotal"] = df.groupby("CustomerID")["Revenue"].transform("sum")
df["CustomerOrders"] = df.groupby("CustomerID")["InvoiceNo"].transform("nunique")
df["ShareOfCustomerSpend"] = (df["Revenue"] / df["CustomerTotal"]).round(4)

df["RevenueZScoreInCountry"] = df.groupby("Country")["Revenue"].transform(
    lambda s: (s - s.mean()) / s.std(ddof=0)
).round(3)

df["CountryMedianRevenue"] = df.groupby("Country")["Revenue"].transform("median")

df[["CustomerID", "Revenue", "CustomerTotal", "CustomerOrders",
    "ShareOfCustomerSpend", "RevenueZScoreInCountry"]].head(8)

,CustomerID,Revenue,CustomerTotal,CustomerOrders,ShareOfCustomerSpend,RevenueZScoreInCountry
0,17850,15.30,5391.21,34,0.0028,-0.016
1,17850,20.34,5391.21,34,0.0038,-0.001
2,17850,22.00,5391.21,34,0.0041,0.004
3,17850,20.34,5391.21,34,0.0038,-0.001
4,17850,20.34,5391.21,34,0.0038,-0.001
5,17850,15.30,5391.21,34,0.0028,-0.016
6,17850,25.50,5391.21,34,0.0047,0.015
7,17850,11.10,5391.21,34,0.0021,-0.029


In [11]:
# Proof that transform preserves shape while agg does not
print("original rows      :", len(df))
print("transform() result :", len(df.groupby("CustomerID")["Revenue"].transform("sum")))
print("agg() result       :", len(df.groupby("CustomerID")["Revenue"].sum()))

original rows      : 397884
transform() result : 397884
agg() result       : 4338


---
## Transformation 6 - Vectorised conditional logic with `np.select()` and `np.where()`

**What:** the same style of tiering as Transformation 4, but done without any Python-level loop.

**Why vectorised:** `np.select()` evaluates boolean masks over whole arrays in C. Conditions are checked in
order and the first match wins, so they must be listed from most to least specific. This is the pattern the
task hint refers to - prefer it whenever the logic can be written as column comparisons.

In [12]:
p90 = df["Revenue"].quantile(0.90)
p50 = df["Revenue"].quantile(0.50)

conditions = [
    df["Revenue"] >= p90,
    df["Revenue"] >= p50,
    df["Revenue"] >= 0,
]
choices = ["Top", "Mid", "Low"]
df["RevenueTier"] = np.select(conditions, choices, default="Unknown")

df["IsWeekend"] = np.where(df["InvoiceDate"].dt.dayofweek >= 5, 1, 0)
df["IsRepeatCustomer"] = np.where(df["CustomerOrders"] > 1, "Repeat", "One-Time")

print(f"P50 = {p50:.2f} | P90 = {p90:.2f}")
print(df["RevenueTier"].value_counts())
print(df["IsRepeatCustomer"].value_counts())

P50 = 11.80 | P90 = 35.40
RevenueTier
Low    197745
Mid    159110
Top     41029
Name: count, dtype: int64
IsRepeatCustomer
Repeat      364363
One-Time     33521
Name: count, dtype: int64


---
## Transformation 7 - Binning with `pd.cut()` and `pd.qcut()`

**What:** convert continuous variables into ordered categories - `cut()` on business-defined edges,
`qcut()` on equal-sized quantiles.

**Why both:** `cut()` gives fixed, interpretable boundaries but uneven bin counts; `qcut()` guarantees equal
population per bin but data-dependent edges. The result is an ordered `Categorical`, which sorts and groups
correctly and uses far less memory than strings.

In [26]:
df["QuantityBucket"] = pd.cut(
    df["Quantity"],
    bins=[0, 1, 5, 10, 20, np.inf],
    labels=["1", "2-5", "6-10", "11-20", "21+"],
)

df["CustomerValueQuartile"] = pd.qcut(
    df["CustomerTotal"], q=4, labels=["Q1-Low", "Q2", "Q3", "Q4-High"]
)

print(df["QuantityBucket"].value_counts().sort_index())
print()
print(df["CustomerValueQuartile"].value_counts().sort_index())
print()
print("dtype:", df["QuantityBucket"].dtype)

QuantityBucket
1         73301
2-5      125275
6-10      73013
11-20     71667
21+       54628
Name: count, dtype: int64

CustomerValueQuartile
Q1-Low     99486
Q2         99609
Q3         99410
Q4-High    99379
Name: count, dtype: int64

dtype: category


---
## Transformation 8 - Custom transformation functions composed with `pipe()`

**What:** two reusable functions - one deriving calendar features, one winsorising outliers - chained with `pipe()`.

**Why `pipe()`:** each function takes a DataFrame and returns a new DataFrame, so they compose into a readable
left-to-right pipeline with no intermediate variables. Because each copies its input, the pipeline is
side-effect free and the same functions can be reused on train/test splits or future data.

In [14]:
def add_time_features(frame: pd.DataFrame, date_col: str = "InvoiceDate") -> pd.DataFrame:
    """Derive calendar features from a datetime column."""
    out = frame.copy()
    dt = out[date_col].dt
    out["Year"] = dt.year
    out["Month"] = dt.month
    out["DayName"] = dt.day_name()
    out["Hour"] = dt.hour
    out["Quarter"] = dt.quarter
    return out


def winsorize(frame: pd.DataFrame, col: str, lower: float = 0.01, upper: float = 0.99) -> pd.DataFrame:
    """Cap extreme values of `col` at the given quantiles instead of dropping them."""
    out = frame.copy()
    lo, hi = out[col].quantile([lower, upper])
    out[f"{col}_Capped"] = out[col].clip(lo, hi)
    return out


def add_basket_features(frame: pd.DataFrame) -> pd.DataFrame:
    """Invoice-level context attached back to every line."""
    out = frame.copy()
    out["BasketSize"] = out.groupby("InvoiceNo")["Quantity"].transform("sum")
    out["BasketLines"] = out.groupby("InvoiceNo")["StockCode"].transform("count")
    out["BasketValue"] = out.groupby("InvoiceNo")["Revenue"].transform("sum")
    return out


df = (
    df.pipe(add_time_features)
      .pipe(winsorize, col="Revenue")
      .pipe(add_basket_features)
)

df[["InvoiceDate", "Year", "Month", "DayName", "Hour",
    "Revenue", "Revenue_Capped", "BasketSize", "BasketValue"]].head()

,InvoiceDate,Year,Month,DayName,Hour,Revenue,Revenue_Capped,BasketSize,BasketValue
0,2010-12-01 08:26:00,2010,12,Wednesday,8,15.30,15.30,40,139.12
1,2010-12-01 08:26:00,2010,12,Wednesday,8,20.34,20.34,40,139.12
2,2010-12-01 08:26:00,2010,12,Wednesday,8,22.00,22.00,40,139.12
3,2010-12-01 08:26:00,2010,12,Wednesday,8,20.34,20.34,40,139.12
4,2010-12-01 08:26:00,2010,12,Wednesday,8,20.34,20.34,40,139.12


---
## Transformation 9 - Method chaining with `assign()`

**What:** create three derived columns in one immutable expression.

**Why `assign()`:** each lambda receives the DataFrame *as it exists at that point in the chain*, so later
columns can depend on earlier ones. It returns a new frame rather than mutating in place, which avoids
`SettingWithCopyWarning` and keeps the whole derivation auditable as a single block.

In [15]:
df = df.assign(
    LogRevenue=lambda f: np.log1p(f["Revenue"]),
    RevenueRankInCustomer=lambda f: f.groupby("CustomerID")["Revenue"].rank(
        method="dense", ascending=False
    ).astype(int),
    IsTopLineForCustomer=lambda f: (f["RevenueRankInCustomer"] == 1),
    RevenuePerUnit=lambda f: (f["Revenue"] / f["Quantity"]).round(2),
)

df[["CustomerID", "Revenue", "LogRevenue",
    "RevenueRankInCustomer", "IsTopLineForCustomer"]].head(8)

,CustomerID,Revenue,LogRevenue,RevenueRankInCustomer,IsTopLineForCustomer
0,17850,15.30,2.791165,22,False
1,17850,20.34,3.060583,17,False
2,17850,22.00,3.135494,15,False
3,17850,20.34,3.060583,17,False
4,17850,20.34,3.060583,17,False
5,17850,15.30,2.791165,21,False
6,17850,25.50,3.277145,13,False
7,17850,11.10,2.493205,26,False


---
## Transformation 10 - Elementwise `DataFrame.map()`

**What:** apply one function to every cell of a numeric sub-frame.

**Why:** `DataFrame.map()` is the whole-table analogue of `Series.map()` (it replaced the old `applymap()`,
which is deprecated / removed in pandas 3.x). Useful for uniform formatting, but it loops in Python, so it
is reserved for presentation frames rather than hot paths.

In [16]:
numeric_view = df[["Quantity", "UnitPrice", "Revenue", "CustomerTotal"]].head(10)
formatted = numeric_view.map(lambda x: f"{x:,.2f}")
formatted

,Quantity,UnitPrice,Revenue,CustomerTotal
0,6.00,2.55,15.30,"5,391.21"
1,6.00,3.39,20.34,"5,391.21"
2,8.00,2.75,22.00,"5,391.21"
3,6.00,3.39,20.34,"5,391.21"
4,6.00,3.39,20.34,"5,391.21"
5,2.00,7.65,15.30,"5,391.21"
6,6.00,4.25,25.50,"5,391.21"
7,6.00,1.85,11.10,"5,391.21"
8,6.00,1.85,11.10,"5,391.21"
9,32.00,1.69,54.08,"3,237.54"


---
## Transformation 11 - Benchmark: `apply()` vs vectorised

Direct evidence for the hint *"avoid `apply()` when a vectorised operation can perform the same task"*.
Both compute exactly the same column three ways.

In [17]:
def timeit(fn, repeats: int = 3) -> float:
    best = float("inf")
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn()
        best = min(best, time.perf_counter() - t0)
    return best


t_rowapply = timeit(lambda: df.apply(lambda r: r["Quantity"] * r["UnitPrice"], axis=1))
t_listcomp = timeit(lambda: [q * p for q, p in zip(df["Quantity"], df["UnitPrice"])])
t_vector   = timeit(lambda: df["Quantity"] * df["UnitPrice"])

results = pd.DataFrame({
    "Method": ["apply(axis=1)", "list comprehension", "vectorised (Series * Series)"],
    "Seconds": [t_rowapply, t_listcomp, t_vector],
})
results["Speedup vs apply"] = (t_rowapply / results["Seconds"]).round(1)
results.round(6)

,Method,Seconds,Speedup vs apply
0,apply(axis=1),2.887947,1.0
1,list comprehension,0.082261,35.1
2,vectorised (Series * Series),0.001001,2885.4


In [18]:
# Sanity check: all three produce identical values
a = df.apply(lambda r: r["Quantity"] * r["UnitPrice"], axis=1)
b = df["Quantity"] * df["UnitPrice"]
print("Identical results:", np.allclose(a, b))

Identical results: True


---
## 12. Summary of the transformed dataset

Every column added above, with the method that produced it.

In [19]:
summary = pd.DataFrame([
    ("Region",                  "Series.map()",           "Country grouped into UK / EU / APAC / Americas / Other"),
    ("Description, Country",    "replace()",              "Placeholder junk standardised or set to NaN"),
    ("PriceBand",               "Series.apply()",         "Budget / Standard / Premium / Luxury from unit price"),
    ("OrderType",               "apply(axis=1)",          "Row-wise label from quantity + price + revenue"),
    ("CustomerTotal",           "groupby().transform()",  "Customer lifetime spend aligned to every row"),
    ("ShareOfCustomerSpend",    "groupby().transform()",  "Line revenue as a fraction of customer total"),
    ("RevenueZScoreInCountry",  "transform(lambda)",      "Revenue standardised within its country"),
    ("RevenueTier",             "np.select()",            "Top / Mid / Low by revenue percentile, vectorised"),
    ("IsWeekend",               "np.where()",             "Weekend flag from the invoice date"),
    ("QuantityBucket",          "pd.cut()",               "Fixed-edge quantity bins"),
    ("CustomerValueQuartile",   "pd.qcut()",              "Equal-population customer spend quartiles"),
    ("Year/Month/DayName/Hour", "custom fn + pipe()",     "Calendar features from a reusable function"),
    ("Revenue_Capped",          "custom fn + pipe()",     "Outliers winsorised at the 1st/99th percentile"),
    ("Basket* columns",         "custom fn + transform()","Invoice-level context attached to each line"),
    ("LogRevenue, ranks",       "assign() chain",         "Log transform and per-customer revenue rank"),
], columns=["Column(s)", "Method", "Meaning"])

summary

,Column(s),Method,Meaning
0,Region,Series.map(),Country grouped into UK / EU / APAC / Americas...
1,"Description, Country",replace(),Placeholder junk standardised or set to NaN
2,PriceBand,Series.apply(),Budget / Standard / Premium / Luxury from unit...
3,OrderType,apply(axis=1),Row-wise label from quantity + price + revenue
4,CustomerTotal,groupby().transform(),Customer lifetime spend aligned to every row
5,ShareOfCustomerSpend,groupby().transform(),Line revenue as a fraction of customer total
6,RevenueZScoreInCountry,transform(lambda),Revenue standardised within its country
7,RevenueTier,np.select(),"Top / Mid / Low by revenue percentile, vectorised"
8,IsWeekend,np.where(),Weekend flag from the invoice date
9,QuantityBucket,pd.cut(),Fixed-edge quantity bins


In [20]:
print(f"Final shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Columns added: {df.shape[1] - raw.shape[1]}")
df.dtypes

Final shape: 397,884 rows x 35 columns
Columns added: 27


InvoiceNo                         object
StockCode                         object
Description                       object
Quantity                           int64
InvoiceDate               datetime64[us]
UnitPrice                        float64
CustomerID                         int64
Country                              str
Revenue                          float64
Region                               str
PriceBand                            str
OrderType                            str
CustomerTotal                    float64
CustomerOrders                     int64
ShareOfCustomerSpend             float64
RevenueZScoreInCountry           float64
CountryMedianRevenue             float64
RevenueTier                          str
IsWeekend                          int64
IsRepeatCustomer                     str
QuantityBucket                  category
CustomerValueQuartile           category
Year                               int32
Month                              int32
DayName         

---
## 13. Export the transformed dataset

In [21]:
Path("output").mkdir(exist_ok=True)

df.to_csv("output/transformed_online_retail.csv", index=False)

# A small sample is what actually goes in the GitHub repo - full file may exceed the 100 MB limit
df.sample(min(5000, len(df)), random_state=42).to_csv(
    "output/transformed_sample.csv", index=False
)

size_mb = Path("output/transformed_online_retail.csv").stat().st_size / 1e6
print(f"Wrote output/transformed_online_retail.csv ({size_mb:.1f} MB)")
print("Wrote output/transformed_sample.csv")

Wrote output/transformed_online_retail.csv (97.1 MB)
Wrote output/transformed_sample.csv


In [27]:
df["FirstPurchaseDate"] = (
    df.groupby("CustomerID")["InvoiceDate"].transform("min")
)

df["DaysSinceFirstPurchase"] = (
    df["InvoiceDate"] - df["FirstPurchaseDate"]
).dt.days

df[[
    "CustomerID",
    "InvoiceDate",
    "FirstPurchaseDate",
    "DaysSinceFirstPurchase"
]].head(10)

,CustomerID,InvoiceDate,FirstPurchaseDate,DaysSinceFirstPurchase
0,17850,2010-12-01 08:26:00,2010-12-01 08:26:00,0
1,17850,2010-12-01 08:26:00,2010-12-01 08:26:00,0
2,17850,2010-12-01 08:26:00,2010-12-01 08:26:00,0
3,17850,2010-12-01 08:26:00,2010-12-01 08:26:00,0
4,17850,2010-12-01 08:26:00,2010-12-01 08:26:00,0
5,17850,2010-12-01 08:26:00,2010-12-01 08:26:00,0
6,17850,2010-12-01 08:26:00,2010-12-01 08:26:00,0
7,17850,2010-12-01 08:28:00,2010-12-01 08:26:00,0
8,17850,2010-12-01 08:28:00,2010-12-01 08:26:00,0
9,13047,2010-12-01 08:34:00,2010-12-01 08:34:00,0


---
## 14. Interview questions

**1. What is the difference between `apply()`, `map()`, and `transform()`?**

- `map()` works on a **Series only** and substitutes values one-to-one from a dict, Series or function. Unmatched
  keys become `NaN`. `DataFrame.map()` is the elementwise whole-table version.
- `apply()` works on a Series (element by element) or a DataFrame along an axis. With `axis=1` the function
  receives a whole row, so it can combine columns. It is the most flexible and the slowest, and the return
  shape is not constrained - it can return a scalar, Series or DataFrame.
- `transform()` applies a function but **must return output with the same shape and index as the input**.
  That constraint is the point: after `groupby().transform()` the result aligns row-for-row with the original
  frame, so it can be assigned back as a column directly. `groupby().agg()` collapses to one row per group instead.

**2. Why are vectorised pandas operations usually preferred?**

Vectorised operations run as compiled C/NumPy loops over contiguous memory blocks, with no Python interpreter
overhead per element and no per-row Series object construction. `apply(axis=1)` builds a Python Series for every
row and calls back into the interpreter each time. The benchmark in Transformation 11 shows the resulting
difference on this dataset. Vectorised code is also shorter, handles `NaN` consistently, and releases the GIL
in many NumPy paths.

**3. When would you use a custom transformation function?**

When the logic is domain-specific, branching, or stateful enough that no built-in expresses it - and especially
when it must be **reused identically** across datasets or across train/test splits. Writing it as a function that
takes a DataFrame and returns a DataFrame (as in Transformation 8) makes it composable with `pipe()`, unit-testable,
and safe to apply to new data later. Inside `groupby().transform()`, a custom lambda also gives group-relative
statistics such as z-scores that no built-in string alias provides.

---
## 15. Conclusion

Eleven advanced transformations were applied to the Online Retail dataset, taking it from eight raw columns to a
fully featured analytical table. The central lesson is **method selection**: `map()` for one-to-one lookups,
`replace()` for standardisation, `transform()` when output must align with the source index, `np.select()` /
`np.where()` when branching logic can be expressed as column masks, and `apply()` reserved for genuinely
row-wise logic. Reusable transformations were written as functions composed with `pipe()` so the pipeline can be
re-run on new data without modification.